# Inspiration

A lot of inspiration for this notebook was taken from [@dmahajanbe23](https://www.kaggle.com/dmahajanbe23).
Massive thanks to him and his great [notebook](https://www.kaggle.com/code/dmahajanbe23/feature-growth-catboost-xgb-ordered-boosting/notebook)!



In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score
from itertools import combinations
from xgboost import XGBClassifier
from scipy.stats import rankdata
from pathlib import Path
import pandas as pd
import numpy as np
import optuna
import gc

# Prepare data

In [ ]:
target = 'Heart Disease'

data_dir = Path('/kaggle/input/playground-series-s6e2/')
df_train = pd.read_csv(data_dir / 'train.csv')
df_test = pd.read_csv(data_dir / 'test.csv')

if not pd.api.types.is_numeric_dtype(df_train[target]):
    df_train[target] = df_train[target].map({'Absence': 0, 'Presence': 1})

X_train = df_train.drop(columns=[target, 'id'])
y_train = df_train[target]

X_test = df_test.drop(columns=['id'])

all_cols = X_train.columns.tolist()

# Feature Engineering

## Frequency encoding

In [ ]:
X_train_col_freq, X_test_col_freq = pd.DataFrame(index=X_train.index), pd.DataFrame(index=X_test.index)
for col in all_cols:
    # freq is a dictionary of "{Col_value: freq}" elements
    freq = X_train[col].value_counts(normalize=True)
    # map col value to freq, Age for example -> 54 becomes 0.074327
    X_train_col_freq[f'{col}_freq'] = X_train[col].map(freq)
    X_test_col_freq[f'{col}_freq'] = X_test[col].map(freq)

X_train_col_freq

## Target encoding

In [ ]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(kfold.split(X_train, y_train))

X_train_target_enc, X_test_target_enc = pd.DataFrame(index=X_train.index), pd.DataFrame(index=X_test.index)

for col in all_cols:
    X_train_target_enc[f'{col}_te'] = 0.0
    for train_idx, val_idx in kfold.split(X_train, y_train):
        col_mean = df_train.iloc[train_idx].groupby(col)[target].mean()
        col_idx = X_train_target_enc.columns.get_loc(f'{col}_te')
        X_train_target_enc.iloc[val_idx, col_idx] = X_train.iloc[val_idx][col].map(col_mean)
    X_test_target_enc[f'{col}_te'] = X_test[col].map(df_train.groupby(col)[target].mean())

X_train_target_enc

# Create new dataframes with the frequency and target encoded dataframes

In [ ]:
X_train = pd.concat([X_train_col_freq, X_train_target_enc], axis=1)
X_test = pd.concat([X_test_col_freq, X_test_target_enc], axis=1)

# Optuna Hyperparameter search

## XGB

In [ ]:
def objective_xgb(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 11),
        'min_child_weight': trial.suggest_float('min_child_weight', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 10.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 500, 8000),
        'max_delta_step': trial.suggest_float('max_delta_step', 0.0, 10.0),
        
        'enable_categorical': True,
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'device': 'cuda',
        'random_state': 42,
        'early_stopping_rounds': 50
    }

    roc_auc_scores = []

    for (tr_idx, val_idx) in folds:
        X_tr = X_train.iloc[tr_idx].copy()
        X_va = X_train.iloc[val_idx].copy()
        
        y_tr = y_train.iloc[tr_idx].copy()
        y_va = y_train.iloc[val_idx].copy()
        
        model_xgb = XGBClassifier(**params)
        model_xgb.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False
        )

        preds = model_xgb.predict_proba(X_va)[:, 1]
        score = roc_auc_score(y_va, preds)
        roc_auc_scores.append(score)

        trial.report(np.mean(roc_auc_scores), step=len(roc_auc_scores))
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(roc_auc_scores)

Important! Make sure to use TPESampler when running Optuna, as search will be optimized instead of random. Pruner is also used to save some time for runs with little potential by cutting them short.

In [ ]:
# sampler = optuna.samplers.TPESampler(seed=43)
# study_xgb = optuna.create_study(direction='maximize', sampler=sampler, pruner=optuna.pruners.MedianPruner())
# study_xgb.optimize(objective_xgb, n_trials=100)

## CatBoost

In [ ]:
def objective_cb(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        
        'eval_metric': 'AUC',
        'od_type': 'Iter',
        'od_wait': 50,
        'task_type': 'GPU',
        'devices': '0',
        'verbose': False,
        'random_seed': 42,
        'use_best_model': True,
    }

    roc_auc_scores = []
    
    for (tr_idx, val_idx) in folds:
        X_tr = X_train.iloc[tr_idx].copy()
        X_va = X_train.iloc[val_idx].copy()
        
        y_tr = y_train.iloc[tr_idx].copy()
        y_va = y_train.iloc[val_idx].copy()

        train_pool = Pool(X_tr, label=y_tr)
        valid_pool = Pool(X_va, label=y_va)
    
        model_cb = CatBoostClassifier(**params)
    
        model_cb.fit(
            train_pool,
            eval_set=valid_pool,
            early_stopping_rounds=100,
            use_best_model=True,
            verbose=False
        )
                
        preds = model_cb.predict_proba(valid_pool)[:, 1]
        roc_auc = roc_auc_score(y_va, preds)
        roc_auc_scores.append(roc_auc)

        trial.report(np.mean(roc_auc_scores), step=len(roc_auc_scores))
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(roc_auc_scores)

In [ ]:
# sampler = optuna.samplers.TPESampler(seed=43)
# study_cb = optuna.create_study(direction='maximize', sampler=sampler, pruner=optuna.pruners.MedianPruner())
# study_cb.optimize(objective_cb, n_trials=100)

## Best XGB & CB params from Optuna

In [ ]:
# XGB
xgb_params = {
    # **study_xgb.best_params,

    # BEST PARAMS FROM A PREVIOUS RUN #1
    'learning_rate': 0.0027846827641948763, 
    'max_depth': 6, 
    'min_child_weight': 10.072367858054276,
    'subsample': 0.5346460796473406,
    'gamma': 2.2746617321067055, 
    'colsample_bytree': 0.920044851410659, 
    'colsample_bylevel': 0.9485092106144615,
    'reg_lambda': 0.06281089611216494, 
    'reg_alpha': 2.0093258572068815,
    'n_estimators': 6080, 
    'max_delta_step': 3.9943642994568944,

    # BEST PARAMS FROM A PREVIOUS RUN #2
    # 'learning_rate': 0.009933014515411013,
    # 'max_depth': 5,
    # 'min_child_weight': 18.038198125076388, 
    # 'subsample': 0.6034456856716826, 
    # 'gamma': 4.044052430754076, 
    # 'colsample_bytree': 0.9967862461970518,
    # 'colsample_bylevel': 0.8414362489042724,
    # 'reg_lambda': 0.1679991415095646, 
    # 'reg_alpha': 0.17484478274588924,
    # 'n_estimators': 3584, 
    # 'max_delta_step': 7.269879919679885,
    
    'enable_categorical': True,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'device': 'cuda',
}
    
# CATBOOST
cb_params = {
    # **study_cb.best_params, 

    # BEST PARAMS FROM A PREVIOUS RUN #1
    'iterations': 432, 
    'depth': 5, 
    'learning_rate': 0.08360408288478018,
    'l2_leaf_reg': 1.9428457210873802, 
    'random_strength': 0.16707719798995085,
    'bagging_temperature': 0.20594833868927714,
    'border_count': 255,

    # BEST PARAMS FROM A PREVIOUS RUN #2
    # 'iterations': 481, 
    # 'depth': 4, 
    # 'learning_rate': 0.07519180261798254,
    # 'l2_leaf_reg': 2.864617326585854, 
    # 'random_strength': 0.6224526970033767,
    # 'bagging_temperature': 0.060836352600343764, 
    # 'border_count': 226,
    
    'loss_function': 'Logloss',
    'task_type': 'GPU',
    'verbose': False,
}

print("BEST XGB PARAMS:\n", xgb_params)
print("BEST CB PARAMS:\n", cb_params)

# Rank-based Stacking

In [ ]:
oof_meta = np.zeros((len(X_train), 4))
test_meta = np.zeros((len(X_test), 4))
oof_rank_mean = np.zeros(len(X_train))
test_rank_mean = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(folds):
    X_tr = X_train.iloc[tr_idx].copy()
    X_va = X_train.iloc[val_idx].copy()
    
    y_tr = y_train.iloc[tr_idx].copy()
    y_va = y_train.iloc[val_idx].copy()
    
    model_cb = CatBoostClassifier(**cb_params, random_seed=(42 + fold))
    train_pool = Pool(X_tr, label=y_tr)
    model_cb.fit(train_pool)
    val_cb = model_cb.predict_proba(X_va)[:, 1]
    te_cb = model_cb.predict_proba(X_test)[:, 1]

    model_xgb = XGBClassifier(**xgb_params, random_state=(42 + fold))
    model_xgb.fit(X_tr, y_tr)
    val_xgb = model_xgb.predict_proba(X_va)[:, 1]
    te_xgb = model_xgb.predict_proba(X_test)[:, 1]

    rank_cb_val  = rankdata(val_cb)  / len(val_cb)
    rank_xgb_val = rankdata(val_xgb) / len(val_xgb)

    rank_cb_te  = rankdata(te_cb)  / len(te_cb)
    rank_xgb_te = rankdata(te_xgb) / len(te_xgb)

    val_meta = np.column_stack([
        rank_cb_val,
        rank_xgb_val,
        (rank_cb_val + rank_xgb_val) / 2,
        np.abs(rank_cb_val - rank_xgb_val)
    ])

    te_meta = np.column_stack([
        rank_cb_te,
        rank_xgb_te,
        (rank_cb_te + rank_xgb_te) / 2,
        np.abs(rank_cb_te - rank_xgb_te)
    ])

    oof_meta[val_idx] = val_meta
    test_meta += te_meta / kfold.n_splits

    oof_rank_mean[val_idx] = val_meta[:, 2]
    test_rank_mean += te_meta[:, 2] / kfold.n_splits

    print(f'Fold {fold+1}: AUC | {roc_auc_score(y_va, val_meta[:, 2])}')
    gc.collect()
    
print(f'Mean Rank CV AUC | {roc_auc_score(y_train, oof_rank_mean)}')

# Optuna Hyperparameter search for xgb_meta model

In [ ]:
def objective_xgb_meta(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 11),
        'min_child_weight': trial.suggest_float('min_child_weight', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 10.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 500, 8000),
        'max_delta_step': trial.suggest_float('max_delta_step', 0.0, 10.0),
        
        'enable_categorical': True,
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'device': 'cuda',
        'random_state': 42,
        'early_stopping_rounds': 50
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    roc_auc_scores = []

    for (tr_idx, val_idx) in cv.split(oof_meta, y_train):
        X_tr = oof_meta[tr_idx].copy()
        X_va = oof_meta[val_idx].copy()
        
        y_tr = y_train.iloc[tr_idx].copy()
        y_va = y_train.iloc[val_idx].copy()
        
        model_xgb_meta = XGBClassifier(**params)
        model_xgb_meta.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False
        )

        preds = model_xgb_meta.predict_proba(X_va)[:, 1]
        score = roc_auc_score(y_va, preds)
        roc_auc_scores.append(score)

        trial.report(np.mean(roc_auc_scores), step=len(roc_auc_scores))
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(roc_auc_scores)


In [ ]:
# sampler = optuna.samplers.TPESampler(seed=1)
# study_xgb_meta = optuna.create_study(direction='maximize', sampler=sampler, pruner=optuna.pruners.MedianPruner())
# study_xgb_meta.optimize(objective_xgb_meta, n_trials=200)

## Best params for xgb_meta

In [ ]:
xgb_meta_params = {
    # **study_xgb_meta.best_params,
    
    # BEST PARAMS FROM A PREVIOUS RUN #1
    'learning_rate': 0.03063930225014738, 
    'max_depth': 9, 
    'min_child_weight': 17.099969444760546,
    'subsample': 0.7270631643925131, 
    'gamma': 6.3093757619367965,
    'colsample_bytree': 0.8543729736823983,
    'colsample_bylevel': 0.7605412494474136,
    'reg_lambda': 0.09450747298443003, 
    'reg_alpha': 0.0010081921221939676,
    'n_estimators': 6411, 
    'max_delta_step': 4.041369351359322,

    # BEST PARAMS FROM A PREVIOUS RUN #2
    # 'learning_rate': 0.03651369513714449,
    # 'max_depth': 8, 
    # 'min_child_weight': 17.260964717982382,
    # 'subsample': 0.6471936168386255,
    # 'gamma': 4.0726891368548985,
    # 'colsample_bytree': 0.892316943687739, 
    # 'colsample_bylevel': 0.8265548013759328,
    # 'reg_lambda': 0.404822889409606,
    # 'reg_alpha': 0.15736197719939526, 
    # 'n_estimators': 5959, 
    # 'max_delta_step': 2.6295583286146633,
    
    'enable_categorical': True,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'device': 'cuda',
    'random_state': 43
}

print("BEST XGB_META PARAMS:\n", xgb_meta_params)

In [ ]:
model_xgb_meta = XGBClassifier(**xgb_meta_params)

model_xgb_meta.fit(oof_meta, y_train)
meta_pred = model_xgb_meta.predict_proba(test_meta)[:, 1]

# Build submission

In [ ]:
preds = (0.5 * test_rank_mean) + (0.5 * meta_pred)

submission = pd.DataFrame({
    'id': df_test['id'],
    target: preds
})

submission.to_csv('submission.csv', index=False)